# Project: Real-Time Password Strength & Breach Analyzer


### Built by: [Pooja]
### Domain: Identity & Access Management (IAM) / Cryptography

## Project Overview
This tool evaluates password security by combining local structural analysis (complexity checks) with live threat intelligence. It queries the **Have I Been Pwned** API to determine if a password has been compromised in a known data breach.

### Key Security Concepts Applied:
* **Pattern Matching:** Uses Regular Expressions (`re`) to enforce structural complexity.
* **Pattern Matching:** Uses Regular Expressions (`re`) to enforce structural complexity.
* **Cryptography & Hashing:** Implements SHA-1 hashing via Python's `hashlib` to protect user data.
* **k-Anonymity Privacy Protocol:** Ensures clear-text passwords and full hashes are never transmitted over the internet.

## Security & Privacy Architecture (k-Anonymity)
To protect user privacy, this tool never sends the actual password over the network. 
1. The password is locally hashed using **SHA-1**.
2. Only the **first 5 characters** of the hash are sent to the API.
3. The API returns a block of all leaked hashes starting with those 5 characters.
4. The script locally checks if the remaining characters match any in the block.

In [1]:
import re
import requests
import hashlib

In [2]:
def hash_and_split_password(password):
    # Convert password to a secure SHA-1 hex string
    sha1_password = hashlib.sha1(password.encode('utf-8')).hexdigest().upper()
    
    # Split the hash into two pieces (k-Anonymity privacy model)
    first5_chars = sha1_password[:5]
    remaining_chars = sha1_password[5:]
    
    return first5_chars, remaining_chars

In [3]:
def check_pwned_api(first5_chars, remaining_chars):
    url = f"https://api.pwnedpasswords.com/range/{first5_chars}"
    response = requests.get(url)
    
    if response.status_code != 200:
        return "Could not connect to breach database."
    
    # Clean and parse the API response to search for a match
    hashes = (line.split(':') for line in response.text.splitlines())
    for target_hash, count in hashes:
        if target_hash == remaining_chars:
            return int(count) # Target found! Returns breach count.
            
    return 0 # Target not found. Password is clean.

In [4]:
def evaluate_complexity(password):
    score = 0
    feedback = []
    
    if len(password) >= 12: score += 2
    elif len(password) >= 8: score += 1
    else: feedback.append("❌ Too short! (12+ characters is ideal).")
        
    if re.search(r"[A-Z]", password): score += 1
    else: feedback.append("❌ Missing an uppercase letter.")
        
    if re.search(r"[a-z]", password): score += 1
    else: feedback.append("❌ Missing a lowercase letter.")
        
    if re.search(r"\d", password): score += 1
    else: feedback.append("❌ Missing a number.")
        
    if re.search(r"[!@#$%^&*(),.?\":{}|<>_]", password): score += 1
    else: feedback.append("❌ Missing a special character.")
        
    return score, feedback

In [5]:
def run_password_audit():
    print("=== CYBERSECURITY PASSWORD AUDIT TOOL ===")
    password = input("Enter a password to test safely: ")
    print("\nAnalyzing across threat intelligence databases...")
    
    # 1. Split hash via Cell 2 function
    first5, remaining = hash_and_split_password(password)
    
    # 2. Check API via Cell 3 function
    try:
        leak_count = check_pwned_api(first5, remaining)
    except Exception:
        leak_count = "Error connecting"
        
    # 3. Check complexity via Cell 4 function
    score, feedback = evaluate_complexity(password)
    
    # 4. Generate Audit Report
    print("\n================ REPORT ================")
    if isinstance(leak_count, int) and leak_count > 0:
        print(f"⚠️ SECURITY ALERT: This password was found in {leak_count:,} public data breaches!")
        print("🔴 STATUS: COMPROMISED (Do not use this password!)")
    else:
        print("🛡️ BREACH STATUS: CLEAN (Not found in known data leaks.)")
        if score == 6:
            print("🟢 RATING: STRONG (Excellent security!)")
        elif 4 <= score < 6:
            print("🟡 RATING: MODERATE (Could be stronger)")
        else:
            print("🔴 RATING: WEAK STRUCTURE")
            
    if feedback:
        print("\nStructure Improvements Needed:")
        for item in feedback:
            print(item)

In [ ]:
run_password_audit()

=== CYBERSECURITY PASSWORD AUDIT TOOL ===
